<a href="https://colab.research.google.com/github/Jess-chege/Data-Science-projects/blob/main/Publication1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#importing the needed libraries
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import librosa
import soundfile as sf
from scipy.signal import wiener
from sklearn.svm import SVC
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix
import torch
from transformers import Wav2Vec2ForCTC



### Training Kenyan audio models to be able to recognize different kenyan accents specifically in kenyan children. https://docs.google.com/document/d/1qhgKvMqlRrKTnlBtQjNPURev1X7tokefvKa3uBzUIfk/edit?usp=sharing

#loading the audio data
def preprocess_audio(file_path):
    audio_signal, sr = librosa.load(file_path, sr=16000)

    # Normalize safely
    max_val = np.max(np.abs(audio))
    if max_val > 0:
        audio = audio_signal / max_val

    # Trim silence
    audio_signal, _ = librosa.effects.trim(audio, top_db=25)

    return audio_signal, sr

In [ ]:
#loading the wav 2wav model
from transformers import Wav2Vec2Processor, Wav2Vec2ForCTC
import torch

processor = Wav2Vec2Processor.from_pretrained("facebook/wav2vec2-base-960h")
model = Wav2Vec2ForCTC.from_pretrained("facebook/wav2vec2-base-960h")


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


preprocessor_config.json:   0%|          | 0.00/159 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json:   0%|          | 0.00/163 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/291 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/85.0 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/378M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/212 [00:00<?, ?it/s]

Wav2Vec2ForCTC LOAD REPORT from: facebook/wav2vec2-base-960h
Key                        | Status  | 
---------------------------+---------+-
wav2vec2.masked_spec_embed | MISSING | 

Notes:
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [ ]:
# using wav2vec processor
inputs = processor(audio_signal, sampling_rate=samplerate, return_tensors="pt", padding=True)

In [ ]:
#getting the logits
with torch.no_grad():
    outputs = model(**inputs)

logits = outputs.logits

In [ ]:
#getting the transcription
predicted_ids = torch.argmax(logits, dim=-1)
prediction = processor.batch_decode(predicted_ids)[0]

print("Prediction:", prediction)

Prediction: BRING MANY SOGS


In [ ]:
#Extracting embeddings
model.config.output_hidden_states = True

with torch.no_grad():
    outputs = model(**inputs)

embeddings = outputs.hidden_states[-1]

In [ ]:
test_audio = '/content/drive/MyDrive/84-121123-0000.flac'

#Loading the test audio
y, sr = librosa.load(test_audio, sr=None)  # sr=None preserves original sampling rate
print(f"Audio loaded: {y.shape[0]} samples, sampling rate: {sr}")

Audio loaded: 33440 samples, sampling rate: 16000


In [ ]:
import torchaudio

waveform, sample_rate = torchaudio.load(test_audio)
print(f"Waveform shape: {waveform.shape}, Sample rate: {sample_rate}")

Waveform shape: torch.Size([1, 33440]), Sample rate: 16000


In [ ]:
# Converting stereo to mono
if len(y.shape) > 1:
    y = np.mean(y, axis=1)

# Optional: normalize
y = y / np.max(np.abs(y))

In [ ]:
# Estimating fundamental frequency (pitch)
f0, voiced_flag, voiced_probs = librosa.pyin(
    y,
    fmin=librosa.note_to_hz('C2'),
    fmax=librosa.note_to_hz('C7')
)

# Calculating intonation score
intonation_score = 1 - (np.nanstd(f0) / np.nanmean(f0))
intonation_score = np.clip(intonation_score, 0, 1)

print(f"Intonation Score: {intonation_score*100:.2f}%")

Intonation Score: 85.45%


In [ ]:
# Calculating amplitude envelope
hop_length = 512
frame_length = 1024
amplitude_envelope = np.array([max(y[i:i+frame_length]) for i in range(0, len(y), hop_length)])

# Detecting silence
threshold = 0.04
silent_frames = np.where(amplitude_envelope < threshold)[0]
total_frames = len(amplitude_envelope)

# Fluency metric: proportion of time speaking vs silence
fluency_score = 1 - (len(silent_frames) / total_frames)
fluency_score = np.clip(fluency_score, 0, 1)

print(f"Fluency Score: {fluency_score*100:.2f}%")

Fluency Score: 39.39%


In [ ]:
pip install jiwer

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 34.8 MB/s eta 0:00:00


In [ ]:
from jiwer import wer, cer

# expected pronounciation
reference_text = "BRING MANY SOCKS"

# Model transcription
predicted_text = "BRING MANY SOGS"

# Computing the Word Error Rate
word_error = wer(reference_text, predicted_text)
print(f"WER: {word_error:.2%}")  # Output as percentage

# Computing the Character Error Rate
char_error = cer(reference_text, predicted_text)
print(f"CER: {char_error:.2%}")  # Output as percentage

WER: 33.33%
CER: 12.50%


The WAV-to-WAV model processes raw audio files directly to evaluate pronunciation and produce transcriptions. Each WAV file is analyzed to generate a predicted text, which is then compared to the expected transcription.

On the dataset, the model achieved a Word Error Rate (WER) of 33.33% and a Character Error Rate (CER) of 12.50%, showing that while most characters were correctly recognized, some words were misrecognized due to variations in pronunciation or pauses in the audio.

This demonstrates a complete WAV-to-WAV workflow, from raw audio input to transcription evaluation, providing a foundation for automated pronunciation assessment.

#### TRAIN THE SVM

WRITE UP
https://docs.google.com/document/d/1oGJTyRxByuptl9MKXaTiK3DqzVTpNAc1_JbjXc_MCdU/edit?usp=sharing

In [ ]:
#claculating the number of audios
num_audios = len([file for file in os.listdir(audio_folder) if file.endswith(".wav")])
print(num_audios)

294


In [ ]:
#calculating the audio duration
duration_seconds = len(audio_files) / samplerate
duration_hours = duration_seconds / 3600
print(duration_seconds)
print(duration_hours)

NameError: name 'audio_files' is not defined

In [ ]:
print(clusters)